# Additional analyses for the revision (items B-E)

Uses the same data, Penman-Monteith SPEI and 1985-2002 vs 2003-2020 window as `SH-ITA_final_PM.ipynb`.

| Section | What it does | Reviewer comment |
|---|---|---|
| **B** | False-discovery control across *all* tests (Benjamini-Hochberg over the 96 class tests and over the 32 IISC/A tests), plus **simulation-calibrated significance levels** from `SH-ITA_synthetic_experiments_v2.ipynb` | R1 #17, #18; R2 #5 |
| **C** | Goodness-of-fit of the gamma distributions used for SPI (Kolmogorov-Smirnov with a parametric-bootstrap p-value), zero-rainfall handling, SPI mean/SD | R2 #6; R1 #20 |
| **D** | Quantitative SPI vs SPEI comparison: correlation, class agreement (kappa), IISC agreement, drought-event frequency and duration | R1 #19 |
| **E** | Medium class split into negative-normal and positive-normal; class assignment by second-half value instead of first-half | R1 #15; R2 #2 |

Needs `reshaped_climate_data.xlsx` and `extra_variables.xlsx` in the same folder. Run time about 5 minutes.

In [ ]:
import os, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
from math import sin, radians, acos, tan, pi
from scipy import stats

# ---- EDIT THESE TWO LINES IF NEEDED -------------------------------------------
DATA = 'reshaped_climate_data.xlsx'   # the workbook your first notebook created (same folder as this notebook)
OUT  = 'sh_ita_outputs'               # folder where tables and figures will be saved
# ------------------------------------------------------------------------------
os.makedirs(OUT, exist_ok=True)
%matplotlib inline

## Settings

In [ ]:
LAT = {'YENAGOA': 4.95, 'IKOT AKPADEN': 4.62, 'IBADAN': 7.38, 'ABUJA': 9.07,
       'JALINGO': 8.89, 'SOKOTO': 13.01, 'GOMBE': 10.28, 'KUKAWA': 12.92}
X_START, X_END = '1982-01-01', '2001-12-31'      # first half  (240 months)
Y_START, Y_END = '2002-01-01', '2021-12-31'      # second half (240 months)
B = 2000                                          # bootstrap replicates
BLOCK = {3: 12, 12: 24}                           # block length (months) by timescale
MIN_N = 10                                        # min class size for a trend label
SEED = 2026
CLIP = 1e-6
COL = {'Low': 'red', 'Medium': 'orange', 'High': 'blue'}   # ONE colour convention for every figure

## Load data

In [ ]:
P = pd.read_excel(DATA, 'Precip_Wide', index_col='Date').drop(columns=['Year', 'Month'], errors='ignore')
T = pd.read_excel(DATA, 'Temp_Wide', index_col='Date').drop(columns=['Year', 'Month'], errors='ignore')
P.index = pd.to_datetime(P.index); T.index = pd.to_datetime(T.index)

## PET (Thornthwaite)

In [ ]:
MID = [15, 46, 75, 105, 135, 166, 196, 227, 258, 288, 319, 349]
def daylength(lat, month):
    decl = radians(23.45 * sin(radians(360 * (284 + MID[month - 1]) / 365.0)))
    x = max(min(-tan(radians(lat)) * tan(decl), 0.999999), -0.999999)
    return 24.0 / pi * acos(x)
def thornthwaite(t, lat):
    clim = t.groupby(t.index.month).mean()
    I = ((clim.clip(lower=0) / 5.0) ** 1.514).sum()
    a = 6.75e-7 * I ** 3 - 7.71e-5 * I ** 2 + 1.792e-2 * I + 0.49239
    out = [16.0 * (10.0 * v / I) ** a * (daylength(lat, d.month) / 12.0) * (d.days_in_month / 30.0)
           if v > 0 else 0.0 for d, v in t.items()]
    return pd.Series(out, index=t.index)
PET = pd.DataFrame({l: thornthwaite(T[l], LAT[l]) for l in P.columns})

## SPI and SPEI per calendar month

In [ ]:
def roll(s, sc): return s.rolling(sc, min_periods=sc).sum()
def spi_cm(p, sc):
    a = roll(p, sc); out = pd.Series(np.nan, index=a.index)
    for m in range(1, 13):
        v = a[a.index.month == m].dropna(); nz = v[v > 0]
        if len(nz) < 10: continue
        q = (v == 0).mean()
        g, _, sc_ = stats.gamma.fit(nz.values, floc=0)
        cdf = np.where(v.values <= 0, q, q + (1 - q) * stats.gamma.cdf(v.values, g, loc=0, scale=sc_))
        out[v.index] = stats.norm.ppf(np.clip(cdf, CLIP, 1 - CLIP))
    return out
def spei_cm(p, pet, sc):
    a = roll(p - pet, sc); out = pd.Series(np.nan, index=a.index)
    for m in range(1, 13):
        v = a[a.index.month == m].dropna(); N = len(v)
        if N < 20: continue
        out[v.index] = stats.norm.ppf((stats.rankdata(v.values) - 0.44) / (N + 0.12))
    return out

## SH-ITA statistics and bootstrap

In [ ]:
def slope(x, y):
    return np.nan if len(x) < 2 or np.ptp(x) == 0 else np.polyfit(x, y, 1)[0]
def ita_stats(x, y):
    xs, ys = np.sort(x), np.sort(y)
    cls = np.where(xs <= -1, 0, np.where(xs >= 1, 2, 1))          # class from FIRST-half value
    r = {}
    for k, nm in enumerate(['low', 'med', 'high']):
        m = cls == k
        r['n_' + nm] = int(m.sum()); r['slope_' + nm] = slope(xs[m], ys[m])
        r['D_' + nm] = (ys[m] - xs[m]).mean() if m.any() else np.nan
    r['IISC'] = (ys.mean() - xs.mean()) / np.abs(xs).mean()
    sl, sh = r['slope_low'], r['slope_high']
    r['ICSC'] = (sh - sl) / abs(sh + sl) if np.isfinite(sl) and np.isfinite(sh) and abs(sh + sl) > 0 else np.nan
    r['A'] = r['D_high'] - r['D_low']
    return r, xs, ys, cls
def cbb(a, L, rng):
    n = len(a); nb = int(np.ceil(n / L))
    return a[(rng.integers(0, n, nb)[:, None] + np.arange(L)).ravel()[:n] % n]
KEYS = ['D_low', 'D_med', 'D_high', 'IISC', 'ICSC', 'A']
def analyse(s, L, seed):
    x = s.loc[X_START:X_END].dropna().values; y = s.loc[Y_START:Y_END].dropna().values
    assert len(x) == len(y), 'halves differ in length: %d vs %d' % (len(x), len(y))
    base, xs, ys, cls = ita_stats(x, y); rng = np.random.default_rng(seed)
    bt = {k: [] for k in KEYS}
    for _ in range(B):
        r, *_ = ita_stats(cbb(x, L, rng), cbb(y, L, rng))
        for k in KEYS: bt[k].append(r[k])
    res = dict(base); res['n'] = len(x)
    for k in KEYS:
        v = np.array(bt[k], float); v = v[np.isfinite(v)]
        res[k + '_lo'], res[k + '_hi'] = (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 100 else (np.nan, np.nan)
    for nm in ['low', 'med', 'high']:
        lo, hi = res['D_%s_lo' % nm], res['D_%s_hi' % nm]
        res['label_' + nm] = '-' if (res['n_' + nm] < MIN_N or not np.isfinite(lo)) else ('Yes(+)' if lo > 0 else 'Yes(-)' if hi < 0 else 'No')
    return res, xs, ys, cls

## Penman-Monteith PET

In [ ]:
def Ra_daily(lat_deg, doy):
    """Extraterrestrial radiation, MJ m-2 day-1 (FAO-56)."""
    phi = np.radians(lat_deg); dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365.0)
    dec = 0.409 * np.sin(2 * np.pi * doy / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    return (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
def es_kpa(t): return 0.6108 * np.exp(17.27 * t / (t + 237.3))
def monthly_from_daily(daily, idx): return pd.Series(np.asarray(daily) * idx.days_in_month.values, index=idx)
def doy_mid(idx): return np.array([pd.Timestamp(d.year, d.month, 15).dayofyear for d in idx])

def pet_hamon(t, lat, kpec=1.0):
    """Hamon (1961): PET[mm/day] = 0.1651 * (N/12) * 216.7 * esat[mb]/(T+273.3) * kpec."""
    N = np.array([daylength(lat, d.month) for d in t.index])
    rhosat = 216.7 * 6.108 * np.exp(17.27 * t.values / (t.values + 237.3)) / (t.values + 273.3)
    return monthly_from_daily(np.where(t.values > 0, 0.1651 * (N / 12.0) * rhosat * kpec, 0.0), t.index)
def pet_oudin(t, lat):
    """Oudin et al. (2005): PET[mm/day] = 0.408 * Ra * (T + 5) / 100 if T + 5 > 0."""
    ra = Ra_daily(lat, doy_mid(t.index))
    return monthly_from_daily(np.where(t.values + 5 > 0, 0.408 * ra * (t.values + 5) / 100.0, 0.0), t.index)
def pet_hargreaves(tmax, tmin, lat):
    """Hargreaves-Samani: PET[mm/day] = 0.0023 * 0.408*Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin), Tmean = (Tmax+Tmin)/2."""
    tm = (tmax + tmin) / 2; ra = Ra_daily(lat, doy_mid(tm.index))
    return monthly_from_daily(0.0023 * 0.408 * ra * (tm.values + 17.8) * np.sqrt(np.clip(tmax.values - tmin.values, 0, None)), tm.index)
def pet_pm_fao56(tmax, tmin, rh, u2, rs, lat, elev):
    """FAO-56 Penman-Monteith on monthly mean data (G = 0). rs in MJ m-2 day-1, u2 m/s at 2 m, rh %."""
    Tx, Tn = tmax.values, tmin.values; T = (Tx + Tn) / 2
    Pk = 101.3 * ((293 - 0.0065 * elev) / 293) ** 5.26; gamma = 0.000665 * Pk
    delta = 4098 * es_kpa(T) / (T + 237.3) ** 2
    es = (es_kpa(Tx) + es_kpa(Tn)) / 2; ea = np.clip(rh.values, 1, 100) / 100.0 * es
    ra = Ra_daily(lat, doy_mid(tmax.index)); rso = (0.75 + 2e-5 * elev) * ra
    rn = 0.77 * rs.values - 4.903e-9 * (((Tx + 273.16) ** 4 + (Tn + 273.16) ** 4) / 2) * (0.34 - 0.14 * np.sqrt(ea)) * (1.35 * np.clip(rs.values / rso, 0.3, 1.0) - 0.35)
    et0 = (0.408 * delta * rn + gamma * 900 / (T + 273) * u2.values * (es - ea)) / (delta + gamma * (1 + 0.34 * u2.values))
    return monthly_from_daily(np.clip(et0, 0, None), tmax.index)

PETS = {'Thornthwaite': PET.copy(),
        'Hamon': pd.DataFrame({l: pet_hamon(T[l], LAT[l]) for l in P.columns}),
        'Oudin': pd.DataFrame({l: pet_oudin(T[l], LAT[l]) for l in P.columns})}
XV = 'extra_variables.xlsx'
if os.path.exists(XV):
    ex = {s: pd.read_excel(XV, s, index_col='Date') for s in ('Tmax', 'Tmin', 'RH', 'Wind', 'Srad')}
    ELEV = pd.read_excel(XV, 'Elevation').set_index('Location')['Elevation_m'].to_dict()
    PETS['Hargreaves'] = pd.DataFrame({l: pet_hargreaves(ex['Tmax'][l], ex['Tmin'][l], LAT[l]) for l in P.columns})
    PETS['Penman-Monteith'] = pd.DataFrame({l: pet_pm_fao56(ex['Tmax'][l], ex['Tmin'][l], ex['RH'][l], ex['Wind'][l], ex['Srad'][l], LAT[l], ELEV[l]) for l in P.columns})
else:
    print('extra_variables.xlsx not found: Hargreaves and Penman-Monteith skipped.')
print('PET formulations:', list(PETS))

In [ ]:
# ---- analysis window for the whole paper (Penman-Monteith needs radiation, available from 1984)
FIT_START = '1984-01-01'
X_START, X_END = '1985-01-01', '2002-12-31'
Y_START, Y_END = '2003-01-01', '2020-12-31'
PM = PETS['Penman-Monteith']
print('First half:', X_START[:7], 'to', X_END[:7], '| second half:', Y_START[:7], 'to', Y_END[:7])

## Comparators: Mann-Kendall (Hamed-Rao), Sen (2017), Alashan (2020)

In [ ]:
def sen_slope(x):
    x = np.asarray(x, float); n = len(x)
    return np.median(np.concatenate([(x[k:] - x[:-k]) / k for k in range(1, n)]))

def mk_test(x, hr=True):
    """Mann-Kendall S, Z, p-value (two-sided) and Sen slope per time step. hr=True applies Hamed & Rao (1998)."""
    x = np.asarray(x, float); n = len(x)
    S = sum(np.sign(x[i + 1:] - x[i]).sum() for i in range(n - 1))
    _, cnt = np.unique(x, return_counts=True)
    var = (n * (n - 1) * (2 * n + 5) - np.sum(cnt * (cnt - 1) * (2 * cnt + 5))) / 18.0
    beta = sen_slope(x); f = 1.0
    if hr:
        r = stats.rankdata(x - beta * np.arange(n)); r = r - r.mean(); den = (r * r).sum()
        lim = stats.norm.ppf(0.975) / np.sqrt(n); sni = 0.0
        for k in range(1, n - 1):
            a = (r[:-k] * r[k:]).sum() / den
            if abs(a) > lim: sni += (n - k) * (n - k - 1) * (n - k - 2) * a
        f = 1 + 2.0 / (n * (n - 1) * (n - 2)) * sni
        f = f if f > 0 else 1.0
    z = 0.0 if S == 0 else (S - np.sign(S)) / np.sqrt(var * f)
    return dict(S=S, Z=z, p=2 * (1 - stats.norm.cdf(abs(z))), sen=beta, vcf=f)

def direction(p, est, alpha=0.05):
    return '0' if not (p < alpha) else ('+' if est > 0 else '-')

In [ ]:
def sen2017_sig(x1, x2, alpha=0.05):
    n = len(x1) + len(x2); s = 2 * (x2.mean() - x1.mean()) / n
    rho = np.corrcoef(np.sort(x1), np.sort(x2))[0, 1]
    sd = np.sqrt(8.0 * np.var(np.concatenate([x1, x2]), ddof=1) * max(1 - rho, 1e-12)) / n
    return abs(s / sd) > stats.norm.ppf(1 - alpha / 2)
def alashan2020_sig(x1, x2, alpha=0.05):
    a, b = np.sort(x1), np.sort(x2); n = len(a) + len(b); s = 2 * (b.mean() - a.mean()) / n
    s1, s2 = a.std(ddof=1), b.std(ddof=1); rho = np.corrcoef(a, b)[0, 1]
    sd = (2.0 / n) * np.sqrt(max(s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2, 1e-12))
    return abs(s / sd) > stats.norm.ppf(1 - alpha / 2)

## Final analysis function (bootstrap p-values + Holm correction)

In [ ]:
BL = {3: (6, 12, 24), 12: (12, 24, 48)}       # block lengths tried; the middle one is primary
ALPHA = 0.05

def boot_p(v):
    """Two-sided percentile-bootstrap p-value for H0: statistic = 0."""
    v = v[np.isfinite(v)]
    return np.nan if len(v) < 100 else max(2 * min((v <= 0).mean(), (v >= 0).mean()), 1.0 / (len(v) + 1))

def holm(pvals):
    """Holm step-down adjustment; NaN entries (untestable classes) are ignored."""
    p = np.array(pvals, float); out = np.full_like(p, np.nan); ok = np.where(np.isfinite(p))[0]
    order = ok[np.argsort(p[ok])]; m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); out[i] = run
    return out

def lab(D, p, n):
    return '-' if (n < MIN_N or not np.isfinite(p)) else ('No' if p >= ALPHA else ('Yes(+)' if D > 0 else 'Yes(-)'))

def analyse_final(s, L, seed, B=B):
    x = s.loc[X_START:X_END].dropna().values; y = s.loc[Y_START:Y_END].dropna().values
    assert len(x) == len(y)
    base, xs, ys, cls = ita_stats(x, y); rng = np.random.default_rng(seed)
    bt = {k: [] for k in KEYS}
    for _ in range(B):
        r, *_ = ita_stats(cbb(x, L, rng), cbb(y, L, rng))
        for k in KEYS: bt[k].append(r[k])
    bt = {k: np.array(v, float) for k, v in bt.items()}
    res = dict(base, n=len(x), L=L)
    for k in KEYS:
        v = bt[k][np.isfinite(bt[k])]
        res[k + '_lo'], res[k + '_hi'] = (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 100 else (np.nan, np.nan)
    names = ['low', 'med', 'high']
    p_raw = [boot_p(bt['D_' + nm]) if base['n_' + nm] >= MIN_N else np.nan for nm in names]
    p_adj = holm(p_raw)
    for i, nm in enumerate(names):
        res['p_raw_' + nm] = p_raw[i]; res['p_holm_' + nm] = p_adj[i]
        res['label_raw_' + nm] = lab(base['D_' + nm], p_raw[i], base['n_' + nm])
        res['label_holm_' + nm] = lab(base['D_' + nm], p_adj[i], base['n_' + nm])
    for k in ('IISC', 'ICSC', 'A'):
        res[k + '_clear'] = bool(np.isfinite(res[k + '_lo']) and (res[k + '_lo'] > 0 or res[k + '_hi'] < 0))
    return res

## Primary analysis with bootstrap p-values (one block length per timescale)

In [ ]:
# Effective significance levels that give a 5% false-positive rate under no change (from SH-ITA_synthetic_experiments_v2.ipynb,
# 'Effective alpha' sheet; phi = 0.6 for 3-month series, 0.9 for 12-month series). UPDATE these after you run the full-size synthetic study.
ALPHA_CAL = {3: dict(cls=0.015, IISC=0.02, A=0.03), 12: dict(cls=0.015, IISC=0.005, A=0.01)}

def analyse_plus(s, L, seed, B=B):
    x = s.loc[X_START:X_END].dropna().values; y = s.loc[Y_START:Y_END].dropna().values; assert len(x) == len(y)
    base, xs, ys, cls = ita_stats(x, y); rng = np.random.default_rng(seed); bt = {k: [] for k in KEYS}
    for _ in range(B):
        r, *_ = ita_stats(cbb(x, L, rng), cbb(y, L, rng))
        for k in KEYS: bt[k].append(r[k])
    bt = {k: np.array(v, float) for k, v in bt.items()}; res = dict(base, n=len(x), L=L); names = ['low', 'med', 'high']
    p_raw = [boot_p(bt['D_' + nm]) if base['n_' + nm] >= MIN_N else np.nan for nm in names]; p_adj = holm(p_raw)
    for i, nm in enumerate(names):
        res['p_raw_' + nm] = p_raw[i]; res['p_holm_' + nm] = p_adj[i]; res['label_holm_' + nm] = lab(base['D_' + nm], p_adj[i], base['n_' + nm])
    for k in ('IISC', 'A', 'ICSC'):
        v = bt[k][np.isfinite(bt[k])]; res['p_' + k] = boot_p(bt[k])
        res[k + '_lo'], res[k + '_hi'] = (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 100 else (np.nan, np.nan)
    return res

order = ['YENAGOA', 'IKOT AKPADEN', 'IBADAN', 'ABUJA', 'JALINGO', 'SOKOTO', 'GOMBE', 'KUKAWA']
rows, SER = [], {}; P84 = P.loc[FIT_START:]
for sc in (3, 12):
    for loc in order:
        for kind in ('SPEI', 'SPI'):
            s_full = spei_cm(P84[loc], PM[loc].loc[FIT_START:], sc) if kind == 'SPEI' else spi_cm(P84[loc], sc)
            SER[(loc, kind, sc)] = s_full
            r = analyse_plus(s_full.loc[X_START:Y_END].dropna(), BL[sc][1], SEED)
            r.update(location=loc.title(), index=f'{kind}-{sc}', scale=sc, kind=kind); rows.append(r)
        print('done', sc, loc, flush=True)
PL = pd.DataFrame(rows); print(len(PL), 'series')

## B. False-discovery control across all tests, and simulation-calibrated levels

In [ ]:
def bh(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); q = np.empty(m); run = 1.0
    for rank, i in zip(range(m, 0, -1), o[::-1]): run = min(run, p[i] * m / rank); q[i] = run
    return q
long = []
for _, r in PL.iterrows():
    for nm in ('low', 'med', 'high'):
        if np.isfinite(r['p_raw_' + nm]):
            long.append(dict(location=r.location, index=r['index'], scale=r.scale, cls=nm, D=r['D_' + nm], p_raw=r['p_raw_' + nm], p_holm=r['p_holm_' + nm]))
CL = pd.DataFrame(long); CL['q_BH_all_tests'] = bh(CL.p_raw.values)
CL['yes_unadjusted'] = CL.p_raw < 0.05; CL['yes_Holm_within_series'] = CL.p_holm < 0.05
CL['yes_BH_all_tests'] = CL.q_BH_all_tests < 0.05
CL['yes_calibrated'] = CL.p_holm < CL.scale.map(lambda s: ALPHA_CAL[s]['cls'])
print('CLASS TESTS (testable cells:', len(CL), 'of 96)')
print(CL.groupby('cls')[['yes_unadjusted', 'yes_Holm_within_series', 'yes_BH_all_tests', 'yes_calibrated']].sum().assign(total=CL.groupby('cls').size()).to_string())
print('all classes:', CL[['yes_unadjusted', 'yes_Holm_within_series', 'yes_BH_all_tests', 'yes_calibrated']].sum().to_dict())
print('\nLabels that survive the calibrated level:')
print(CL[CL.yes_calibrated][['location', 'index', 'cls', 'D', 'p_holm']].round(3).to_string(index=False))
PL['q_IISC_BH'] = bh(PL.p_IISC.values); PL['q_A_BH'] = bh(PL.p_A.values)
PL['IISC_calibrated'] = PL.p_IISC < PL.scale.map(lambda s: ALPHA_CAL[s]['IISC']); PL['A_calibrated'] = PL.p_A < PL.scale.map(lambda s: ALPHA_CAL[s]['A'])
print('\nOVERALL TESTS over 32 series:  IISC  p<0.05:', int((PL.p_IISC < 0.05).sum()), '| BH<0.05:', int((PL.q_IISC_BH < 0.05).sum()), '| calibrated:', int(PL.IISC_calibrated.sum()))
print('                                 A     p<0.05:', int((PL.p_A < 0.05).sum()), '| BH<0.05:', int((PL.q_A_BH < 0.05).sum()), '| calibrated:', int(PL.A_calibrated.sum()))
print('IISC significant under calibrated level:'); print(PL[PL.IISC_calibrated][['location', 'index', 'IISC', 'p_IISC']].round(3).to_string(index=False))

## C. Goodness-of-fit of the SPI gamma distributions

In [ ]:
NSIM = 100; rng_g = np.random.default_rng(SEED); gof = []
for sc in (3, 12):
    for loc in order:
        a = roll(P84[loc], sc)
        for m in range(1, 13):
            v = a[a.index.month == m].dropna(); nz = v[v > 0]
            row = dict(location=loc.title(), scale=sc, month=m, n=len(v), zero_share=round(float((v == 0).mean()), 3), n_nonzero=len(nz))
            if len(nz) < 10: row.update(KS=np.nan, p_KS=np.nan, status='undefined (<10 non-zero)')
            else:
                g_, _, sc_ = stats.gamma.fit(nz.values, floc=0); D0 = stats.kstest(nz.values, 'gamma', args=(g_, 0, sc_)).statistic; cnt = 0
                for _ in range(NSIM):
                    sim = rng_g.gamma(g_, sc_, len(nz)); gs, _, ss = stats.gamma.fit(sim, floc=0)
                    cnt += stats.kstest(sim, 'gamma', args=(gs, 0, ss)).statistic >= D0
                row.update(KS=D0, p_KS=(1 + cnt) / (NSIM + 1), status='fitted')
            gof.append(row)
GOF = pd.DataFrame(gof)
f = GOF[GOF.status == 'fitted']
print('Calendar-month gamma fits:', len(f), '| undefined months:', int((GOF.status != 'fitted').sum()))
print('Share of fits rejected at p<0.05 (about 5% expected by chance):', round(float((f.p_KS < 0.05).mean()), 3), '| by scale:', f.groupby('scale').apply(lambda g: round(float((g.p_KS < 0.05).mean()), 3)).to_dict())
print('Fits rejected at p<0.01:', int((f.p_KS < 0.01).sum()))
print('\nUndefined months (too few non-zero totals):'); print(GOF[GOF.status != 'fitted'].groupby(['location', 'scale']).month.apply(list).to_string())
mom = [dict(location=loc.title(), index=f'SPI-{sc}', mean=SER[(loc, 'SPI', sc)].mean(), sd=SER[(loc, 'SPI', sc)].std()) for sc in (3, 12) for loc in order]
SPIMOM = pd.DataFrame(mom).round(3); print('\nSPI mean and SD over the fitted record (should be near 0 and 1):'); print(SPIMOM.describe().loc[['min', 'max']].to_string())

## D. Quantitative SPI vs SPEI comparison

In [ ]:
def cls3(v): return np.where(v <= -1, 0, np.where(v >= 1, 2, 1))
def events(v):
    m = (v <= -1).astype(int); runs = []; c = 0
    for z in m:
        if z: c += 1
        elif c: runs.append(c); c = 0
    if c: runs.append(c)
    return len(runs), (float(np.mean(runs)) if runs else 0.0)
rows = []; PLi = PL.set_index(['location', 'index'])
for sc in (3, 12):
    for loc in order:
        spi = SER[(loc, 'SPI', sc)].loc[X_START:Y_END]; spe = SER[(loc, 'SPEI', sc)].loc[X_START:Y_END]
        j = pd.concat([spi, spe], axis=1, keys=['spi', 'spe']).dropna(); ca, cb = cls3(j.spi.values), cls3(j.spe.values)
        po = (ca == cb).mean(); pe = sum((ca == k).mean() * (cb == k).mean() for k in range(3)); kappa = (po - pe) / (1 - pe)
        rs, ri = PLi.loc[(loc.title(), f'SPI-{sc}')], PLi.loc[(loc.title(), f'SPEI-{sc}')]
        d = lambda r: ('+' if r.IISC > 0 else '-') if (r.IISC_lo > 0 or r.IISC_hi < 0) else '0'
        row = dict(location=loc.title(), scale=sc, r_SPI_SPEI=round(np.corrcoef(j.spi, j.spe)[0, 1], 3), class_agreement=round(po, 3), kappa=round(kappa, 3),
                   IISC_SPI=round(rs.IISC, 2), IISC_SPEI=round(ri.IISC, 2), IISC_difference=round(ri.IISC - rs.IISC, 2), IISC_dir_SPI=d(rs), IISC_dir_SPEI=d(ri), same_dir=d(rs) == d(ri))
        for nm, ser in (('SPI', spi), ('SPEI', spe)):
            for half, (a_, b_) in (('first', (X_START, X_END)), ('second', (Y_START, Y_END))):
                v = ser.loc[a_:b_].values; v = v[np.isfinite(v)]; ne, dur = events(v)
                row[f'{nm}_{half}_pct_dry'] = round(100 * float((v <= -1).mean()), 1); row[f'{nm}_{half}_events'] = ne; row[f'{nm}_{half}_mean_duration'] = round(dur, 1)
        rows.append(row)
SPISPEI = pd.DataFrame(rows)
print('Correlation SPI-SPEI (median by scale):', SPISPEI.groupby('scale').r_SPI_SPEI.median().to_dict(), '| class agreement (median):', SPISPEI.groupby('scale').class_agreement.median().to_dict())
print('Same overall direction/significance in SPI and SPEI:', int(SPISPEI.same_dir.sum()), '/ 16')
print('Median |IISC(SPEI) - IISC(SPI)|:', SPISPEI.groupby('scale').IISC_difference.apply(lambda s: round(float(s.abs().median()), 2)).to_dict())
SPISPEI[['location', 'scale', 'r_SPI_SPEI', 'class_agreement', 'kappa', 'IISC_SPI', 'IISC_SPEI', 'same_dir', 'SPI_first_pct_dry', 'SPI_second_pct_dry', 'SPEI_first_pct_dry', 'SPEI_second_pct_dry']]

## E. Medium-class split and class-assignment sensitivity

In [ ]:
def class_labels(x, y, scheme, by, L, seed, Bn=1000):
    def masks(ref):
        if scheme == '3': return {'low': ref <= -1, 'med': (ref > -1) & (ref < 1), 'high': ref >= 1}
        return {'low': ref <= -1, 'neg_normal': (ref > -1) & (ref <= 0), 'pos_normal': (ref > 0) & (ref < 1), 'high': ref >= 1}
    def Ds(xv, yv):
        xs_, ys_ = np.sort(xv), np.sort(yv); M = masks(xs_ if by == 'x' else ys_)
        return {k: ((ys_[m] - xs_[m]).mean() if m.sum() >= MIN_N else np.nan) for k, m in M.items()}, {k: int(m.sum()) for k, m in M.items()}
    base, n = Ds(x, y); rng = np.random.default_rng(seed); bt = {k: [] for k in base}
    for _ in range(Bn):
        d, _ = Ds(cbb(x, L, rng), cbb(y, L, rng))
        for k in bt: bt[k].append(d[k])
    p_raw = [boot_p(np.array(bt[k], float)) if np.isfinite(base[k]) else np.nan for k in base]; p_adj = holm(p_raw)
    return {k: lab(base[k], p_adj[i], n[k]) for i, k in enumerate(base)}, base, n

rows = []
for sc in (3, 12):
    for loc in order:
        for kind in ('SPEI', 'SPI'):
            s = SER[(loc, kind, sc)].loc[X_START:Y_END].dropna(); L = BL[sc][1]
            x = s.loc[X_START:X_END].values; y = s.loc[Y_START:Y_END].values
            lx, dx, nx = class_labels(x, y, '3', 'x', L, SEED); ly, dy, ny = class_labels(x, y, '3', 'y', L, SEED); l4, d4, n4 = class_labels(x, y, '4', 'x', L, SEED)
            row = dict(location=loc.title(), index=f'{kind}-{sc}')
            for k in ('low', 'med', 'high'): row[f'3class_by_first_{k}'] = lx[k]; row[f'3class_by_second_{k}'] = ly[k]
            for k in ('low', 'neg_normal', 'pos_normal', 'high'): row[f'4class_{k}'] = l4[k]; row[f'4class_{k}_D'] = round(d4[k], 3) if np.isfinite(d4[k]) else np.nan; row[f'4class_{k}_n'] = n4[k]
            rows.append(row)
CLS = pd.DataFrame(rows)
agree = sum((CLS[f'3class_by_first_{k}'] == CLS[f'3class_by_second_{k}']).sum() for k in ('low', 'med', 'high'))
flip = sum(((CLS[f'3class_by_first_{k}'].str.startswith('Yes')) & (CLS[f'3class_by_second_{k}'].str.startswith('Yes')) & (CLS[f'3class_by_first_{k}'] != CLS[f'3class_by_second_{k}'])).sum() for k in ('low', 'med', 'high'))
print(f'Class assignment by first-half vs second-half value: identical labels in {agree}/96 class cells; opposite-sign Yes/Yes: {flip}')
nn, pn = CLS['4class_neg_normal'], CLS['4class_pos_normal']
print('Medium class split: negative-normal vs positive-normal labels')
print(pd.crosstab(nn, pn, rownames=['neg-normal'], colnames=['pos-normal']).to_string())
mixed = CLS[(nn.str.startswith('Yes')) & (pn.str.startswith('Yes')) & (nn != pn)]
print('series where the two normal sub-classes move in opposite directions:', len(mixed))
hid = CLS[(CLS['3class_by_first_med'] == 'No') & ((nn.str.startswith('Yes')) | (pn.str.startswith('Yes')))]
print('series where the 3-class medium label is No but a normal sub-class is Yes:', len(hid)); print(hid[['location', 'index', '4class_neg_normal', '4class_pos_normal']].to_string(index=False))

## Export

In [ ]:
with pd.ExcelWriter(os.path.join(OUT, 'SH-ITA_additional_analyses.xlsx'), engine='openpyxl') as w:
    CL.round(4).to_excel(w, sheet_name='B class tests FDR', index=False)
    PL[['location', 'index', 'IISC', 'IISC_lo', 'IISC_hi', 'p_IISC', 'q_IISC_BH', 'IISC_calibrated', 'A', 'A_lo', 'A_hi', 'p_A', 'q_A_BH', 'A_calibrated']].round(4).to_excel(w, sheet_name='B overall tests FDR', index=False)
    GOF.round(4).to_excel(w, sheet_name='C gamma GOF', index=False)
    SPIMOM.to_excel(w, sheet_name='C SPI mean SD', index=False)
    SPISPEI.to_excel(w, sheet_name='D SPI vs SPEI', index=False)
    CLS.to_excel(w, sheet_name='E class split and assignment', index=False)
print('saved to', OUT)